In [0]:
source_path = "/Volumes/workspace/cyberaccess/source_data"

bronze_schema = "workspace.cyberaccess_bronze"

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {bronze_schema}
""")

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

auth_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("timestamp", TimestampType(), False),
    StructField("ip_address", StringType(), True),
    StructField("country", StringType(), True),
    StructField("city", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("authentication_method", StringType(), True),
    StructField("login_status", StringType(), True),
    StructField("failure_reason", StringType(), True)
])

In [0]:
from pyspark.sql.types import BooleanType, IntegerType

privilege_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("timestamp", TimestampType(), False),
    StructField("previous_role", StringType(), True),
    StructField("new_role", StringType(), True),
    StructField("previous_privilege_level", IntegerType(), True),
    StructField("new_privilege_level", IntegerType(), True),
    StructField("change_reason", StringType(), True),
    StructField("approved", BooleanType(), True)
])

In [0]:
access_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("timestamp", TimestampType(), False),
    StructField("device_id", StringType(), True),
    StructField("application_id", StringType(), True),
    StructField("action", StringType(), True),
    StructField("resource", StringType(), True),
    StructField("access_status", StringType(), True)
])

In [0]:
auth_df = (
    spark.read
    .schema(auth_schema)
    .json(f"{source_path}/authentication_events.json")
)

In [0]:
access_df = (
    spark.read
    .schema(access_schema)
    .json(f"{source_path}/access_events.json")
)

In [0]:
privilege_df = (
    spark.read
    .schema(privilege_schema)
    .json(f"{source_path}/privilege_events.json")
)

In [0]:
auth_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"{bronze_schema}.authentication_events"
    )

In [0]:
access_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"{bronze_schema}.access_events"
    )

In [0]:
privilege_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"{bronze_schema}.privilege_events"
    )

In [0]:
spark.sql(f"SHOW TABLES IN {bronze_schema}").show()

In [0]:
display(
    spark.sql(f"""
        SELECT *
        FROM {bronze_schema}.authentication_events
        LIMIT 10
    """)
)